# Part C — Project report template

**This is the notebook your group submits.** Rename it with your group
identifier. This is a **short exploratory project**, not a comprehensive
research study. Aim for two or three well-chosen model cases and one
result that you can explain clearly.

## Assessment

The project is graded **G (Godkänt)** or **U (Underkänt)**. There are no
points or weighted criteria. A passing report adequately addresses:

- a focused question, prediction, and controlled design;
- appropriate quantitative evidence;
- a physical interpretation of the main result;
- awareness of relevant limitations and enough information to reproduce
  the cases;
- a clear notebook and contribution statement.

Keep the scope small. Use one main independent variable, and make sure
the submitted notebook runs from top to bottom.


## Group and research question

**Group members:**  

**Research question:**


## Prediction

State your expected result **before** presenting simulation results. Name
the physical mechanism or scaling that supports it.

**Prediction:**


## Model scope

This one-layer shallow-water model is hydrostatic and depth averaged. In
the default linear configuration it assumes small surface displacement.
All cells remain wet; it does not represent breaking, run-up, inundation,
or a resolved bottom boundary layer. Rayleigh damping is an idealized
energy-loss parameter.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from shallowwater import (
    ModelParams, backend_info, compute_dt_cfl, depth_on_u,
    load_bathymetry, make_grid, make_wind_forcing_from_file,
    run_model, shelf_bathymetry, uniform_wind_forcing, zero_forcing,
)

print(backend_info())


## Reusable experiment function

The function below makes the controls explicit. `depth` may be a positive
scalar, an `(Ny, Nx)` array, or a function that creates such an array from
the grid. Initial-state choices are `cross_pulse`, `gaussian`, and `rest`.
Uniform wind is controlled by `wind_x`, `wind_y`, `wind_ramp_hours`, and
`wind_off_hours`. Rotation is controlled by the constant Coriolis
parameter `f` in $\mathrm{s^{-1}}$; use `f=0.0` for no rotation.

If you use a custom external file, the submitted notebook must explain
how it can be reproduced. Prefer generating arrays in this notebook or
using the supplied course files; do not use an unexplained absolute path.


In [ ]:
def make_initial_state(
    grid, params, *, kind="cross_pulse", amplitude=0.08,
    radius=60e3, x_fraction=0.25, y_fraction=0.50,
):
    x0, y0 = x_fraction * grid.Lx, y_fraction * grid.Ly
    eta = np.zeros((grid.Ny, grid.Nx))
    u = np.zeros((grid.Ny, grid.Nx + 1))
    v = np.zeros((grid.Ny + 1, grid.Nx))
    X, Y = np.meshgrid(grid.x_c, grid.y_c)

    if kind == "rest":
        return eta, u, v
    if kind == "gaussian":
        eta = amplitude * np.exp(-((X-x0)**2 + (Y-y0)**2) / radius**2)
        return eta, u, v
    if kind == "cross_pulse":
        eta_line = amplitude * np.exp(-((grid.x_c-x0) / radius)**2)
        eta = np.repeat(eta_line[None, :], grid.Ny, axis=0)
        H_u = depth_on_u(grid, params.H)
        eta_u = amplitude * np.exp(-((grid.x_u-x0) / radius)**2)
        u = np.repeat(eta_u[None, :], grid.Ny, axis=0) * np.sqrt(params.g/H_u)
        return eta, u, v
    raise ValueError("kind must be 'cross_pulse', 'gaussian', or 'rest'")


def run_case(
    label,
    *,
    Nx=120, Ny=24, Lx=1.2e6, Ly=240e3,
    depth=400.0, f=0.0, damping=0.0,
    initial_kind="cross_pulse", initial_amplitude=0.08,
    initial_radius=60e3, initial_x_fraction=0.25,
    initial_y_fraction=0.50,
    wind_x=0.0, wind_y=0.0, wind_ramp_hours=1.0,
    wind_off_hours=None, wind_file=None,
    tmax_hours=5.0,
):
    grid = make_grid(Nx, Ny, Lx, Ly)
    H = depth(grid) if callable(depth) else depth
    if isinstance(H, (str, Path)):
        H = load_bathymetry(H, grid)
    params = ModelParams(
        H=H, g=9.81, f0=f, beta=0.0,
        r=damping, linear=True,
    )
    dt = compute_dt_cfl(grid, params, cfl=0.42)

    if wind_file is not None:
        envelope = lambda t: min(1.0, t/(wind_ramp_hours*3600))
        forcing = make_wind_forcing_from_file(wind_file, grid, envelope=envelope)
    elif wind_x != 0.0 or wind_y != 0.0:
        t_off = None if wind_off_hours is None else wind_off_hours * 3600
        forcing = lambda t, g, p: uniform_wind_forcing(
            t, g, p, tau_x=wind_x, tau_y=wind_y,
            t_ramp=wind_ramp_hours*3600, t_off=t_off,
        )
    else:
        forcing = zero_forcing

    initial = lambda g, p: make_initial_state(
        g, p, kind=initial_kind, amplitude=initial_amplitude,
        radius=initial_radius, x_fraction=initial_x_fraction,
        y_fraction=initial_y_fraction,
    )
    out = run_model(
        tmax=tmax_hours*3600, dt=dt, grid=grid, params=params,
        forcing_fn=forcing, ic_fn=initial, save_every=5,
        out_vars=("eta",),
    )
    print(
        f"{label}: Nx={Nx}, Ny={Ny}, Lx={Lx/1e3:.0f} km, "
        f"dx={grid.dx/1e3:.1f} km, dt={dt:.1f} s, "
        f"f={f:.2e} s^-1, frames={len(out['time'])}"
    )
    return {"label": label, "grid": grid, "params": params, "out": out}


## Experiment

Run one baseline and at least one controlled variation. Change one main
parameter and keep the other relevant settings fixed. Add your own code
cells below; the choice of plots and diagnostic is part of the project.

Briefly state which parameter you changed and which controls you kept
fixed.


## Results and interpretation

Show the evidence needed to answer the question. Include at least one
quantitative result—not only an animation—and explain the main physical
pattern you observe. Label figures with units. Briefly mention the single
most relevant model, resolution, or experimental caveat for your result.


## Conclusion

Answer the research question directly in a short paragraph. State
whether the prediction was supported and cite the main numerical result.


## Contributions and submission check

**Member contributions:**  

Before submission:

- [ ] All group members are named.
- [ ] The evidence needed for the conclusion is visible.
- [ ] The kernel was restarted and all cells ran in order without error.
- [ ] The notebook is reproducible and contains no unexplained local path.
